## Plot bracket results 

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

### Figure 9

In [ ]:
from matplotlib.patches import FancyArrowPatch

def add_double_arrow(ax, start_x, end_x, y_position, label, shift_text_y = 0.02):
    # Create a double arrow to represent the interval
    arrow_length = 0.2  # Adjust this for arrow length
    arrow = FancyArrowPatch((start_x, y_position), (end_x, y_position),
                             arrowstyle='<|-|>', color='black', lw=1,
                             mutation_scale=10, shrinkA= 0.0, shrinkB= 0.0)
    ax.add_patch(arrow)
    
    # Add the label in the middle of the arrow
    ax.text((start_x + end_x) / 2, y_position + shift_text_y, label, ha='center', va='bottom', fontsize=6)

    return ax

def add_iteration_arrows(ax, model_calls_cumsum, n_EI, n_EM, ypos = None, ymin = None, shift_text_y = 0.04):
    ylim = ax.get_ylim()
    delta_y = ylim[1]-ylim[0]
    shift = 0.075

    if ypos is None:
        ypos = ylim[0]-shift*delta_y

    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[0], end_x=model_calls_cumsum[n_EI], y_position=ypos, label = "EI", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI], end_x=model_calls_cumsum[n_EI+n_EM], y_position=ypos, label = "EM", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI+n_EM], end_x=model_calls_cumsum[-1], y_position=ypos, label = "TR", shift_text_y = shift_text_y)
    
    if ymin is None:
        ax.set_ylim([ylim[0]-1.5*shift*delta_y, ylim[1]])
    else:
        ax.set_ylim([ymin, ylim[1]])
    
    return ax

In [ ]:
# Load the vars
import matplotlib.pyplot as plt
import pickle 
import numpy as np 
from utils.plotting import add_vline
import seaborn as sns 

data_file = "_experiments/RESS/bracket_paper/bracket_data.pkl"
name = "Bracket"

# Load the data
loaded_objects = []
# Load results
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects
ref_x, ref_q = model.load_opt(measure = args.measure, alpha = args.alpha, tau = args.tau)
ref_x = np.atleast_1d(ref_x)

DOE_calls = np.sum(np.array(ro_data.doe["model_calls"]))
EI_calls = np.sum(np.array(ro_data.doe["model_calls"] + ro_data.EI_infill["model_calls"]))
EM_calls = np.sum(np.array(ro_data.doe["model_calls"] + ro_data.EI_infill["model_calls"] + ro_data.EM_infill["model_calls"]))
TR_calls = np.sum(np.array(ro_data.doe["model_calls"] + ro_data.EI_infill["model_calls"] + ro_data.EM_infill["model_calls"] + ro_data.TR_infill["model_calls"]))

best_x_list = [ro_data.doe["cand_x"]] + ro_data.EI_infill["best_x"] + ro_data.EM_infill["best_x"] + ro_data.TR_infill["best_x"]
cand_x_list = [ro_data.doe["cand_x"]] + ro_data.EI_infill["cand_x"] + ro_data.EM_infill["cand_x"] + ro_data.TR_infill["cand_x"]
best_J_list = [ro_data.doe["cand_J"]] + ro_data.EI_infill["best_J"] + ro_data.EM_infill["best_J"] + ro_data.TR_infill["best_J"]
best_J_std_list = [ro_data.doe["cand_J_std"]] + ro_data.EI_infill["best_J_std"] + ro_data.EM_infill["best_J_std"]

model_calls = ro_data.doe["model_calls"] + ro_data.EI_infill["model_calls"] + ro_data.EM_infill["model_calls"] + ro_data.TR_infill["model_calls"]

best_x = np.array(best_x_list)
cand_x = np.array(cand_x_list)
best_J = np.array(best_J_list)
best_J_std = np.array(best_J_std_list)
model_calls_cumsum = np.cumsum(np.array(model_calls))

n_EI = ro_data.EI_infill["n"]
n_EM = ro_data.EM_infill["n"]
n_TR = ro_data.TR_infill["n"]

x_err = model_calls_cumsum[:n_EI + n_EM + 1]
y_err = best_J[:n_EI + n_EM + 1]
y_std = best_J_std[:n_EI + n_EM + 1]

linewidth = 1.25


fig, ax = plt.subplots(1, 2, figsize=np.array([300, 150]) / 72.27)

# ------------------------------------------------------------------
# Left subplot
# ------------------------------------------------------------------
J_line, = ax[0].plot(
    model_calls_cumsum,
    best_J,
    label=r"Best $J$",
    linewidth=linewidth,
)

std_patch = ax[0].fill_between(
    x_err,
    y_err - y_std,
    y_err + y_std,
    alpha=0.3,
    color=J_line.get_color(),
    label=r"$\pm$ std",
)

ref_line, = ax[0].plot(
    model_calls_cumsum,
    np.ones(len(model_calls_cumsum)) * ref_q.numpy(),
    linestyle="--",
    color="grey",
)

ax[0].set_xlabel("Model calls")
ax[0].set_ylabel("$q_{95}$")
ax[0] = add_iteration_arrows(
    ax[0], model_calls_cumsum, n_EI, n_EM, shift_text_y=1.5
)
add_vline(
    ax=ax[0],
    n_doe=DOE_calls,
    n_EI=EI_calls,
    n_EM=EM_calls,
    n_TR=TR_calls,
)

# Legend for the blue line, shaded area, and dashed gray line
ax[0].legend(
    handles=[J_line, std_patch],
    labels=["$\mu_{\\widetilde{Q}_{\\tau}}$", "$\sigma_{\\widetilde{Q}_{\\tau}}$"],
    loc="lower center",
    bbox_to_anchor=(0.5, 1.1),
    ncol=2,
    frameon=True,
    borderaxespad=0,
    borderpad=0.2,
    handlelength=1.3,      # was 0.6
    handletextpad=0.35,    # was 0.2
    columnspacing=1.0,     # was 0.4
    labelspacing=0.35,     # was 0.2
    fontsize=8,
)



# ------------------------------------------------------------------
# Right subplot
# ------------------------------------------------------------------
labels = [f"$x_{i}$" for i in range(1, best_x.shape[1] + 1)]

# These are the solid lines that should appear in the legend
lines = ax[1].plot(
    model_calls_cumsum,
    best_x,
    linewidth=linewidth,
)

# for cand_points in cand_x.T:
#     ax[1].scatter(model_calls_cumsum, cand_points, s = 3, marker = "s", alpha = 0.2)


for line, label in zip(lines, labels):
    line.set_label(label)

# Reference lines are deliberately not included in the legend
for i, line in enumerate(lines):
    ax[1].plot(
        model_calls_cumsum,
        np.ones(len(model_calls_cumsum)) * ref_x[i],
        linestyle="--",
        color=line.get_color(),
        linewidth=linewidth,
        label="_nolegend_",
    )

ax[1].set_xlabel("Model calls")
ax[1].set_ylabel("$x_i$")
ax[1] = add_iteration_arrows(
    ax[1], model_calls_cumsum, n_EI, n_EM, shift_text_y=1.5
)
add_vline(
    ax=ax[1],
    n_doe=DOE_calls,
    n_EI=EI_calls,
    n_EM=EM_calls,
    n_TR=TR_calls,
)

# Only the solid lines are passed to the legend.
# With six solid lines, ncol=3 produces two rows.
ax[1].legend(
    handles=lines,
    labels=labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.1),
    ncol=3,
    frameon=True,
    borderaxespad=0,
    borderpad=0.2,
    handlelength=0.82,     # shorten the line samples
    handletextpad=0.25,   # reduce gap between line and label
    columnspacing=0.7,    # reduce gap between columns
    labelspacing=0.3,
    fontsize=8,
)


sns.despine(ax=ax[0])
sns.despine(ax=ax[1])

fig.text(
    0.03, 0.05, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.51, 0.05, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

# Reserve space above the axes for the legends
fig.tight_layout()

output_folder = "_plots/RESS"
tag = f"{name}_conv"
fig.savefig(f'{output_folder}/{tag}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0, dpi = 400)

### Plot deterministic and robust optimum

Rerun fenicsx

In [ ]:
from models.library import Bracket
from models.bracket import create_mesh, fem_simulation, compute_stress
import torch 
import numpy as np 

def sample_simulation(s_d, xi_sample):
    msh_data = create_mesh(x=s_d.tolist(), p=model.p.tolist(), circle_radius=model.circle_radius, h=model.h, offset=model.offset)
    domain = msh_data.mesh
    num_elements = domain.topology.index_map(domain.topology.dim).size_global

    sig_v_sample = np.zeros((n_samp_q, num_elements))

    n = len(xi_sample)
    checkpoints = {max(1, int(n * k / 10)): f"{k*10}%" for k in range(1, 11)}

    for i, xi in enumerate(xi_sample):
        if i+1 in checkpoints:
            print(checkpoints[i+1])
        uh, domain = fem_simulation(msh_data,model.p[-1],xi[0],xi[1],model.circle_radius)
        sig, sig_v = compute_stress(domain, uh)
        sig_v_sample[i] = sig_v.x.array.real

    return sig_v_sample, domain

model = Bracket(design_uq=False)
n_samp_q = int(1e2)
xi_sample = model.xi_dist.sample(n_samp_q)

# Robust opt
s_d_robust = torch.tensor([83.00, 48.28, 9.93, 60.96, 29.56])
sig_v_sample_robust, domain_robust = sample_simulation(s_d_robust, xi_sample)

# Deterministic opt
s_d_det = torch.tensor([100.0, 50.0, 15.67, 60.0, 11.43])
sig_v_sample_det, domain_det = sample_simulation(s_d_det, xi_sample)

In [ ]:
from dolfinx import plot
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
from models.bracket import plot_von_mises_matplotlib
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import pickle 

mean_det = np.mean(sig_v_sample_det, axis=0)
mean_robust = np.mean(sig_v_sample_robust, axis=0)
mean_vmin = min(mean_det.min(), mean_robust.min())
mean_vmax = max(mean_det.max(), mean_robust.max())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2)

_, ax[0], tpc00, data_det = plot_von_mises_matplotlib(
    mean_det, domain_det, fig_ax=(fig, ax[0]), vmin=mean_vmin, vmax=mean_vmax
)
_, _, tpc01, data_rob = plot_von_mises_matplotlib(
    mean_robust, domain_robust, fig_ax=(fig, ax[1]), vmin=mean_vmin, vmax=mean_vmax
)

cax0 = inset_axes(ax[1], width="4%", height="70%", loc='center left',
                  bbox_to_anchor=(1.05, 0, 1, 1),
                  bbox_transform=ax[1].transAxes, borderpad=0)


with open(f"_experiments/Bracket/mono_level_high_q/von_mises_data_det.pkl", "wb") as f:
    pickle.dump(data_det, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(f"_experiments/Bracket/mono_level_high_q/von_mises_data_rob.pkl", "wb") as f:
    pickle.dump(data_rob, f, protocol=pickle.HIGHEST_PROTOCOL)

fig.colorbar(tpc01, cax=cax0, label="von Mises Stress")

## Plot directly from data

In [ ]:
from pathlib import Path
import pickle
import numpy as np 

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

folder = Path("_experiments/RESS/bracket_paper")

# Only load pickle files you trust.
with open(folder / "von_mises_data_det.pkl", "rb") as f:
    data_det = pickle.load(f)

with open(folder / "von_mises_data_rob.pkl", "rb") as f:
    data_rob = pickle.load(f)

# Both files were saved with the same color limits.
vmin = data_det["vmin"]
vmax = data_det["vmax"]

fig, ax = plt.subplots(1, 2, figsize=np.array([235, 125])/72.27)

for a, data, title in zip(
    ax,
    [data_det, data_rob],
    ["Deterministic design", "Robust design"],
):
    tpc = a.tripcolor(
        data["triang"],
        facecolors=data["field"],
        edgecolors="face",
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
    )

    a.set_aspect("equal")
    a.set_xticks([])
    a.set_yticks([])
    # a.set_title(title)

    for spine in a.spines.values():
        spine.set_visible(False)

# Shared colorbar to the right of the second subplot.
cax = inset_axes(
    ax[1],
    width="4%",
    height="90%",
    loc="center left",
    bbox_to_anchor=(1.05, 0, 1, 1),
    bbox_transform=ax[1].transAxes,
    borderpad=0,
)

fig.colorbar(tpc, cax=cax, label="von Mises Stress (MPa)")

# Reserve room for the colorbar and its label.
fig.subplots_adjust(right=0.85, wspace=0.15)

# Optional:
# fig.savefig("von_mises_comparison.pdf", bbox_inches="tight")


fig.text(
    0.0, 0.02, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.48, 0.02, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)


fig.tight_layout()

name = "bracket"
output_folder = "_plots/RESS"
tag = f"{name}_rob_vs_det"
fig.savefig(f'{output_folder}/{tag}.png', format = 'png', bbox_inches = 'tight', pad_inches = 0.0, dpi = 400)